# Konwersja głosu RVC

Praca inżynierska. Klaudiusz Waśko, Informatyka stosowana, Politechnika Krakowska.

`Runtime` → `Change runtime type` → **GPU**, potem komórki po kolei.


In [ ]:
#@title 1. Dysk Google (zalecane, opcjonalne)
#@markdown Uruchom przed aplikacją, jeśli chcesz zapis modelu na Dysku. Możesz pominąć i pobrać pliki w aplikacji.

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
dest = Path("/content/drive/MyDrive/RVC_Colab_Models")
dest.mkdir(parents=True, exist_ok=True)
print(f"Dysk podłączony. Modele: {dest}")


In [ ]:
#@title 2. Instalacja
#@markdown Pobranie kodu, interfejsu i Applio. Pierwszy raz trwa kilkanaście minut.

import os
import subprocess
import sys

os.chdir("/content")

REPO_URL = "https://github.com/waskok/rvc-voice-tool-pl.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/rvc-voice-tool-pl"
APPLIO_TAG = "3.6.5"
APPLIO_DIR = "/content/Applio"
APPLIO_VENV = "/content/applio-venv"


def say(msg):
    print(msg, flush=True)


def run(cmd, cwd=None, env=None):
    proc = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if proc.returncode != 0:
        tail = "\n".join(proc.stdout.splitlines()[-40:])
        if tail:
            print(tail)
        raise RuntimeError("Nie powiódł się krok: " + " ".join(cmd[:3]))
    return proc.stdout


gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    stdout=subprocess.PIPE,
    stderr=subprocess.DEVNULL,
    text=True,
)
name = gpu.stdout.strip().splitlines()[0] if gpu.returncode == 0 and gpu.stdout.strip() else ""
say("GPU: " + name if name else "Brak GPU. Ustaw Runtime, Change runtime type, GPU i uruchom komórkę ponownie.")

say("Pobieranie kodu narzędzia...")
subprocess.run(["rm", "-rf", REPO_DIR], check=False)
run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, REPO_DIR])
if not os.path.isfile(os.path.join(REPO_DIR, "app", "main.py")):
    raise FileNotFoundError("Brak app/main.py po pobraniu kodu.")
say("Kod narzędzia: gotowe")

say("Instalacja interfejsu...")
run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")])
say("Interfejs: gotowe")

say("Instalacja Applio 3.6.5...")
run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
req = os.path.join(APPLIO_DIR, "requirements.txt")
if not os.path.isfile(req):
    subprocess.run(["rm", "-rf", APPLIO_DIR], check=False)
    run(["git", "clone", "--depth", "1", "--branch", APPLIO_TAG, "https://github.com/IAHispano/Applio.git", APPLIO_DIR])
subprocess.run(["rm", "-rf", APPLIO_VENV], check=False)
run([sys.executable, "-m", "virtualenv", "-p", "python3", APPLIO_VENV])
py = os.path.join(APPLIO_VENV, "bin", "python")
run([py, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([py, "-m", "pip", "install", "-q", "-r", req])
say("Applio: gotowe")

say("Pobieranie modeli pomocniczych...")
env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
run([py, "core.py", "prerequisites"], cwd=APPLIO_DIR, env=env)
say("Modele pomocnicze: gotowe")

check = subprocess.run(
    [py, "-c", "import torch; print('tak' if torch.cuda.is_available() else 'nie')"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.DEVNULL,
    text=True,
)
sees = "tak" if "tak" in check.stdout else "nie"
say("Applio widzi GPU: " + sees)
say("Instalacja zakończona. Uruchom następną komórkę.")


In [ ]:
#@title 3. Uruchom aplikację
#@markdown Zostaw tę komórkę włączoną. Kliknij link, który pojawi się poniżej.

import os
import re
import signal
import subprocess
import sys

from IPython.display import HTML, display

if "REPO_DIR" not in globals() or "APPLIO_DIR" not in globals() or "APPLIO_VENV" not in globals():
    raise RuntimeError("Najpierw uruchom komórkę instalacji.")

env = os.environ.copy()
env["APPLIO_DIR"] = APPLIO_DIR
env["APPLIO_PYTHON"] = os.path.join(APPLIO_VENV, "bin", "python")
env["APP_SHARE"] = "1"
env["APP_HOST"] = "0.0.0.0"
env["APP_PORT"] = "7860"
env["MPLBACKEND"] = "Agg"
env["PYTHONUNBUFFERED"] = "1"


def _stop(proc):
    if proc.poll() is not None:
        return
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait(timeout=5)


print("Uruchamiam aplikację...", flush=True)
proc = subprocess.Popen(
    [sys.executable, "-u", "-m", "app.main"],
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    start_new_session=True,
)
shown = False
stopped = False
tail = []
try:
    for line in proc.stdout:
        clean = re.sub(r"\x1b\[[0-9;]*m", "", line)
        tail.append(clean.rstrip())
        del tail[:-30]
        match = re.search(r"https://\S*gradio\.live\S*", clean)
        if match and not shown:
            url = match.group(0).rstrip(").,]}>\"'")
            shown = True
            display(HTML(
                '<p style="font-size:1.05rem;margin:12px 0 4px">Kliknij, aby przejść do aplikacji:</p>'
                f'<p style="font-size:1.2rem;margin:0"><a href="{url}" target="_blank">{url}</a></p>'
            ))
except KeyboardInterrupt:
    stopped = True
finally:
    if stopped:
        _stop(proc)

if stopped:
    print("Zatrzymano.", flush=True)
elif not shown:
    print("Aplikacja nie wystartowała.", flush=True)
    if tail:
        print("\n".join(tail))
elif proc.poll() is not None:
    print("Aplikacja zatrzymała się. Uruchom komórkę ponownie.", flush=True)
